# 🛠️ BƯỚC 1: TRÍCH XUẤT VECTOR ĐẶC TRƯNG & TẠO DATABASE
> **Đồ án:** Hệ thống Nhận diện Khuôn mặt & Điểm danh Sinh viên  
> **Mô tả:** Notebook này dùng để đọc ảnh sinh viên từ thư mục `dataset/`, trích xuất vector đặc trưng 512 chiều bằng model **InsightFace (ArcFace)**, sau đó lưu thành dữ liệu `database.pkl`.

---

### 1. Khai báo thư viện và Khởi tạo Mô hình AI
* **InsightFace (`buffalo_sc`):** Mô hình Deep Learning rút gọn, tối ưu cho CPU laptop nhưng vẫn đảm bảo độ chính xác cao.

In [4]:
# %%
import os
import cv2
import pickle
import numpy as np
import insightface
from insightface.app import FaceAnalysis

# Khởi tạo mô hình InsightFace
# name='buffalo_sc' là bản model gọn nhẹ, chạy rất nhanh trên CPU laptop
app = FaceAnalysis(name='buffalo_sc', providers=['CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

print("✅ Đã tải xong model InsightFace thành công!")

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\PC/.insightface\models\buffalo_sc\det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\PC/.insightface\models\buffalo_sc\w600k_mbf.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
✅ Đã tải xong model InsightFace thành công!


---
### 2. Trích xuất Vector và Lưu Database (`database.pkl`)
* **Quy trình:**
  1. Duyệt qua từng thư mục sinh viên trong `dataset/`.
  2. Phát hiện khuôn mặt trong từng bức ảnh.
  3. Lấy vector đặc trưng 512D (`normed_embedding`).
  4. Tính vector trung bình cho mỗi sinh viên (nếu có nhiều ảnh) để tăng độ chính xác.
  5. Đóng gói và lưu dữ liệu dưới dạng file `database.pkl`.

In [5]:
dataset_dir = "dataset"
known_embeddings = {} # Dictionary chứa: { "MãSV_Tên": vector_512D }

# Duyệt qua từng thư mục sinh viên trong dataset
for student_folder in os.listdir(dataset_dir):
    folder_path = os.path.join(dataset_dir, student_folder)
    
    if not os.path.isdir(folder_path):
        continue
        
    print(f"🔄 Đang xử lý dữ liệu cho: {student_folder}...")
    vectors = []
    
    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)
        img = cv2.imread(img_path)
        
        if img is None:
            continue
            
        # Nhận diện khuôn mặt trong ảnh
        faces = app.get(img)
        
        if len(faces) > 0:
            # Lấy vector đặc trưng (normed_embedding)
            embedding = faces[0].normed_embedding
            vectors.append(embedding)
        else:
            print(f"   ⚠️ Không tìm thấy mặt trong ảnh: {img_name}")
            
    # Tính vector trung bình nếu sinh viên có nhiều ảnh
    if len(vectors) > 0:
        mean_vector = np.mean(vectors, axis=0)
        mean_vector = mean_vector / np.linalg.norm(mean_vector) # Chuẩn hóa lại vector
        known_embeddings[student_folder] = mean_vector
        print(f"   └─ ✅ Đã trích xuất xong cho {student_folder} ({len(vectors)} ảnh hợp lệ)")

# Lưu dữ liệu dictionary ra file pickle
with open("database.pkl", "wb") as f:
    pickle.dump(known_embeddings, f)

print("\n🎉 HOÀN THÀNH! File 'database.pkl' đã được tạo thành công.")

🔄 Đang xử lý dữ liệu cho: HoangNMSE200102...

🎉 HOÀN THÀNH! File 'database.pkl' đã được tạo thành công.
